# NYC Yellow Taxi Commercial Analytics — Reproducible Demo

This notebook demonstrates the core analysis workflow using a **100,000-row sample of real NYC TLC Yellow Taxi data** from January 2025.

The full portfolio project analyses 75M+ trips from January 2025 through July 2026. The sample is included so the analysis can be reproduced without downloading the full dataset.


In [ ]:
!pip -q install duckdb
import duckdb
import pandas as pd

con = duckdb.connect()
sample_path = "yellow_tripdata_2025-01_sample_100k.parquet"

con.execute(f"""
CREATE OR REPLACE VIEW yellow_taxi AS
SELECT *
FROM read_parquet('{sample_path}');
""")

print("Sample loaded successfully.")

## 1. Basic data audit


In [ ]:
con.sql("""
SELECT
    COUNT(*) AS trips,
    MIN(tpep_pickup_datetime) AS earliest_pickup,
    MAX(tpep_pickup_datetime) AS latest_pickup,
    COUNT(*) FILTER (WHERE trip_distance <= 0) AS non_positive_distance,
    COUNT(*) FILTER (WHERE total_amount <= 0) AS non_positive_total
FROM yellow_taxi
""")

## 2. Commercial revenue KPIs

Payment types 3 (No Charge) and 4 (Dispute) are kept in the raw data but excluded from the main commercial KPI view. This avoids treating these transaction categories as ordinary paid commercial trips.


In [ ]:
con.sql("""
SELECT
    COUNT(*) AS commercial_trips,
    ROUND(SUM(total_amount), 2) AS recorded_revenue,
    ROUND(AVG(total_amount), 2) AS avg_revenue_per_trip,
    ROUND(AVG(fare_amount), 2) AS avg_fare
FROM yellow_taxi
WHERE payment_type NOT IN (3, 4)
""")

## 3. Hourly demand and revenue


In [ ]:
hourly = con.sql("""
SELECT
    EXTRACT(HOUR FROM tpep_pickup_datetime) AS pickup_hour,
    COUNT(*) AS trips,
    ROUND(SUM(total_amount), 2) AS revenue,
    ROUND(AVG(total_amount), 2) AS avg_revenue_per_trip
FROM yellow_taxi
WHERE payment_type NOT IN (3, 4)
GROUP BY 1
ORDER BY 1
""").df()

hourly

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(hourly['pickup_hour'], hourly['revenue'], marker='o')
plt.xlabel('Pickup hour')
plt.ylabel('Recorded revenue ($)')
plt.title('Sample: Yellow Taxi Revenue by Pickup Hour')
plt.xticks(range(24))
plt.grid(True, alpha=0.3)
plt.show()

## 4. Reproducing the full project

The complete analysis uses the official NYC TLC Yellow Taxi Trip Records for January 2025 through July 2026. The full raw dataset is not stored in this repository because it is over 1 GB. Instead, this repository includes a real 100,000-row sample and the analytical code.

Source: NYC Taxi & Limousine Commission (TLC) Trip Record Data.
